# Data Cleaning — shoppingqueriesdataset.xlsx

This notebook performs data cleaning on the custom ESCI dataset:
- Load the dataset
- Drop junk columns
- Fix invalid labels (misaligned rows)
- Strip whitespace
- Remove duplicates
- Validate label distribution
- Save cleaned dataset as CSV for training

In [15]:
import pandas as pd
import os
from collections import Counter

## 1. Load the Dataset

In [16]:
DATASET_DIR = r"c:\Moi\Thesis\Code\RetailTalkFolder\shopping_queries_dataset"
FILE_PATH = os.path.join(DATASET_DIR, "shoppingqueriesdataset.xlsx")

df = pd.read_excel(FILE_PATH, engine='openpyxl')
print(f"Original shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)

Original shape: (52643, 3)
Columns: ['query', 'product_title', 'label']


,query,product_title,label
0,summer dress,Boho Floral V-Neck Summer Dress for Women,E
1,summer dress,Casual Sleeveless Sun Dress with Pockets,E
2,summer dress,Lightweight Cotton Maxi Skirt,S
3,summer dress,Wide Brim Straw Sun Hat,C
4,summer dress,Organic Extra Virgin Olive Oil 500ml,I
5,running shoes,Nike Air Zoom Pegasus Running Shoes,E
6,running shoes,Adidas Ultraboost Lightweight Sneakers,S
7,running shoes,Performance Moisture-Wicking Athletic Socks,C
8,running shoes,Stainless Steel Non-Stick Frying Pan,I
9,iphone charger,Apple 20W USB-C Power Adapter Wall Charger,E


## 2. Drop Junk Columns

In [17]:
# Drop any 'Unnamed' columns (artifacts from Excel)
unnamed_cols = [col for col in df.columns if col.startswith('Unnamed')]
if unnamed_cols:
    print(f"Dropping junk columns: {unnamed_cols}")
    df = df.drop(columns=unnamed_cols)
else:
    print("No junk columns found.")

print(f"Shape after drop: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

No junk columns found.
Shape after drop: (52643, 3)
Columns: ['query', 'product_title', 'label']


## 3. Strip Whitespace

In [18]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

# Uppercase labels for consistency
df['label'] = df['label'].str.upper()

print("Whitespace stripped and labels uppercased.")

Whitespace stripped and labels uppercased.


## 3.5 Remove Repeated Filler Text from Queries

Some queries have repeated padding like `"Set Pack Set Pack Set Pack Set"` appended.
This step detects and removes those repeated trailing patterns.

In [19]:
import re

def remove_repeated_filler(text):
    """
    Detects repeated trailing word patterns like:
      'Folder Long White 10 Pack Tagboard White Set Pack Set Pack Set Pack Set'
    and strips them down to:
      'Folder Long White 10 Pack Tagboard White'
    
    Algorithm: find the shortest repeating tail unit (1-4 words) that repeats 2+ times.
    """
    if not isinstance(text, str):
        return text
    
    words = text.split()
    
    # Try pattern lengths of 1 to 4 words
    for pattern_len in range(1, 5):
        if len(words) < pattern_len * 3:
            continue
        
        # Get the last N words as candidate pattern
        candidate = words[-pattern_len:]
        
        # Count how many times this pattern repeats at the end
        repeat_count = 0
        pos = len(words)
        while pos >= pattern_len:
            chunk = words[pos - pattern_len : pos]
            if chunk == candidate:
                repeat_count += 1
                pos -= pattern_len
            else:
                break
        
        # If pattern repeats 3+ times, it's filler — remove all but the first occurrence
        if repeat_count >= 3:
            clean_words = words[:pos + pattern_len]  # keep one occurrence
            return ' '.join(clean_words)
    
    return text

# Apply to query column
print("Before cleaning — sample queries:")
for q in df['query'].head(5).tolist():
    print(f"  '{q}'")

df['query'] = df['query'].apply(remove_repeated_filler)

print("\nAfter cleaning — same queries:")
for q in df['query'].head(5).tolist():
    print(f"  '{q}'")

Before cleaning — sample queries:
  'summer dress'
  'summer dress'
  'summer dress'
  'summer dress'
  'summer dress'

After cleaning — same queries:
  'summer dress'
  'summer dress'
  'summer dress'
  'summer dress'
  'summer dress'


## 4. Check for Invalid Labels

In [20]:
# Also apply to product_title column in case it has the same issue
df['product_title'] = df['product_title'].apply(remove_repeated_filler)

# Show how many queries were cleaned
original_df = pd.read_excel(FILE_PATH, engine='openpyxl')
for col_name in ['query']:
    if col_name in original_df.columns:
        original_df[col_name] = original_df[col_name].astype(str).str.strip()
changed = (original_df['query'].astype(str).str.strip() != df['query']).sum()
print(f"Queries cleaned (filler removed): {changed} out of {len(df)}")

Queries cleaned (filler removed): 0 out of 52643


In [21]:
valid_labels = {'E', 'S', 'C', 'I'}
invalid_mask = ~df['label'].isin(valid_labels)
invalid_count = invalid_mask.sum()

print(f"Invalid label rows: {invalid_count}")
if invalid_count > 0:
    print("\nRows with invalid labels:")
    display(df[invalid_mask])

Invalid label rows: 0


In [22]:
# Remove rows with invalid labels (misaligned data)
if invalid_count > 0:
    df = df[~invalid_mask].copy()
    print(f"Removed {invalid_count} rows with invalid labels.")
    print(f"Shape after fix: {df.shape}")
else:
    print("All labels are valid.")

All labels are valid.


## 5. Check for Nulls

In [23]:
null_counts = df.isnull().sum()
print("Null counts per column:")
print(null_counts)

if null_counts.sum() > 0:
    print(f"\nDropping {null_counts.sum()} rows with nulls...")
    df = df.dropna(subset=['query', 'product_title', 'label'])
    print(f"Shape after dropping nulls: {df.shape}")
else:
    print("No nulls found.")

Null counts per column:
query            0
product_title    0
label            0
dtype: int64
No nulls found.


## 6. Remove Duplicates

In [24]:
num_duplicates = df.duplicated().sum()
print(f"Duplicate rows: {num_duplicates}")

if num_duplicates > 0:
    df = df.drop_duplicates()
    print(f"Removed {num_duplicates} duplicates.")
    print(f"Shape after dedup: {df.shape}")
else:
    print("No duplicates found.")

Duplicate rows: 450
Removed 450 duplicates.
Shape after dedup: (52193, 3)


## 7. Dataset Summary

In [25]:
print(f"Final shape: {df.shape}")
print(f"Unique queries: {df['query'].nunique()}")
print(f"Unique products: {df['product_title'].nunique()}")
print(f"\nLabel distribution:")

label_names = {'E': 'Exact', 'S': 'Substitute', 'C': 'Complement', 'I': 'Irrelevant'}
label_counts = df['label'].value_counts()
for label in ['E', 'S', 'C', 'I']:
    count = label_counts.get(label, 0)
    pct = count / len(df) * 100
    print(f"  {label} ({label_names[label]}): {count} ({pct:.1f}%)")

Final shape: (52193, 3)
Unique queries: 10250
Unique products: 14861

Label distribution:
  E (Exact): 12571 (24.1%)
  S (Substitute): 12906 (24.7%)
  C (Complement): 13014 (24.9%)
  I (Irrelevant): 13702 (26.3%)


In [26]:
# Show sample rows per label
for label in ['E', 'S', 'C', 'I']:
    print(f"\n--- {label} ({label_names[label]}) sample ---")
    display(df[df['label'] == label].sample(min(5, len(df[df['label'] == label])), random_state=42))


--- E (Exact) sample ---


,query,product_title,label
45215,beefpcpc,Argentina Beef Loaf 150g,E
8516,liver spread sachet,Reno Brand Liver Spread sachet,E
39952,chok nut,Choc Nut King Size Peanut Milk Chocolate 24s,E
1410,glass food storage containers with lids,Pyrex 18-Piece Glass Storage Set,E
22395,hall's candy,Halls Menthol Candy 10s,E



--- S (Substitute) sample ---


,query,product_title,label
48678,umbrella fold,Long Umbrella Black 1pc,S
35734,body scrub,Silka Papaya Body Scrub 200g,S
18740,hansel mocha 10s,Rebisco Sandwich Mocha 10s,S
34246,push pins,Joy Thumbtacks Silver 50pcs,S
19594,fudgee barr 10s,Quake Overload Cake Bar,S



--- C (Complement) sample ---


,query,product_title,label
18868,sales invoice book,Carbon Paper,C
10805,food storage containers airtight,Arm & Hammer Fridge Fresh Baking Soda Refriger...,C
17019,binder clip 3/4 inch,Notes,C
7892,jolly mushroom 400,Nestle All Purpose Cream,C
44098,emergencypc,Eveready Battery AA 2pcs,C



--- I (Irrelevant) sample ---


,query,product_title,label
29586,magic sarap,Sunsilk Green Shampoo 180ml,I
19400,nescafe 3 in 1,Safeguard Bar Soap 130g,I
15709,milo chocolate drink,Zonrox Bleach 500ml,I
22906,mang tomas sauce,Colgate Toothbrush,I
8221,pamatay uhaw,Bench Daily Spell Body Mist,I


## 8. Save Cleaned Dataset

In [27]:
# Save as CSV (for build_from_csv.py)
CSV_OUTPUT = os.path.join(
    r"c:\Moi\Thesis\Code\RetailTalkFolder\classification_identification\custom_esci",
    "custom_esci_template.csv"
)
df.to_csv(CSV_OUTPUT, index=False)
print(f"Saved cleaned CSV to: {CSV_OUTPUT}")
print(f"Rows: {len(df)}")

# Also save as Excel for reference
XLSX_OUTPUT = os.path.join(DATASET_DIR, "shoppingqueriesdataset_cleaned.xlsx")
df.to_excel(XLSX_OUTPUT, index=False, engine='openpyxl')
print(f"Saved cleaned Excel to: {XLSX_OUTPUT}")

Saved cleaned CSV to: c:\Moi\Thesis\Code\RetailTalkFolder\classification_identification\custom_esci\custom_esci_template.csv
Rows: 52193
Saved cleaned Excel to: c:\Moi\Thesis\Code\RetailTalkFolder\shopping_queries_dataset\shoppingqueriesdataset_cleaned.xlsx


In [28]:
# Verify saved CSV
df_verify = pd.read_csv(CSV_OUTPUT)
print(f"Verified CSV shape: {df_verify.shape}")
print(f"Remaining duplicates: {df_verify.duplicated().sum()}")
print(f"Invalid labels: {(~df_verify['label'].isin(valid_labels)).sum()}")

Verified CSV shape: (52193, 3)
Remaining duplicates: 0
Invalid labels: 0
